# Export Fine-Tuned Gemma 2 for Ollama Deployment

This notebook merges the LoRA adapters into the base model and exports a GGUF file
that can be loaded into Ollama for local inference.

**Steps:**
1. Mount Google Drive (adapter weights are there)
2. Merge LoRA adapters into base model
3. Convert to GGUF format (quantized Q4_K_M)
4. Download the GGUF file to your laptop

**Runtime:** Use a T4 GPU (free tier is fine)

## Step 1: Install Dependencies & Mount Drive

In [ ]:
!pip install -q transformers peft accelerate bitsandbytes torch
!pip install -q gguf sentencepiece protobuf

from google.colab import drive
drive.mount('/content/drive')

## Step 2: Load Base Model + LoRA Adapter and Merge

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

BASE_MODEL = "google/gemma-2-2b-it"
# Use v2 adapter (trained on 363 examples - best fine-tuned version)
ADAPTER_PATH = "/content/drive/MyDrive/mi-therapy-capstone/adapters/mi-therapy-gemma2-v2"
MERGED_PATH = "/content/gemma2-mi-merged"

print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

print(f"Loading LoRA adapter from {ADAPTER_PATH}...")
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)

print("Merging LoRA weights into base model...")
merged_model = model.merge_and_unload()

print(f"Saving merged model to {MERGED_PATH}...")
merged_model.save_pretrained(MERGED_PATH)
tokenizer.save_pretrained(MERGED_PATH)

print("Merge complete!")

## Step 3: Quick Sanity Test (Optional)

In [ ]:
# Quick test to verify the merged model works
from transformers import pipeline

pipe = pipeline("text-generation", model=merged_model, tokenizer=tokenizer, max_new_tokens=100)

test_messages = [
    {"role": "user", "content": "I don't think I have a problem with drinking. Everyone does it."}
]

prompt = tokenizer.apply_chat_template(test_messages, tokenize=False, add_generation_prompt=True)
output = pipe(prompt, do_sample=True, temperature=0.7, top_p=0.9)
print("Test response:")
print(output[0]['generated_text'][len(prompt):])

## Step 4: Convert to GGUF Format

This creates a quantized GGUF file (~1.5GB) that Ollama can load.

In [ ]:
# Clone llama.cpp for the conversion script
!git clone --depth 1 https://github.com/ggerganov/llama.cpp /content/llama.cpp
!pip install -q -r /content/llama.cpp/requirements/requirements-convert_hf_to_gguf.txt

In [ ]:
# Convert to GGUF with Q4_K_M quantization (good balance of quality vs size)
!python /content/llama.cpp/convert_hf_to_gguf.py \
    /content/gemma2-mi-merged \
    --outfile /content/gemma2-mi-therapist.gguf \
    --outtype q4_K_M

import os
size_gb = os.path.getsize('/content/gemma2-mi-therapist.gguf') / (1024**3)
print(f"\nGGUF file size: {size_gb:.2f} GB")

## Step 5: Copy to Google Drive for Download

In [ ]:
import shutil

DRIVE_OUTPUT = "/content/drive/MyDrive/mi-therapy-capstone/gguf/gemma2-mi-therapist.gguf"
os.makedirs(os.path.dirname(DRIVE_OUTPUT), exist_ok=True)

print("Copying GGUF to Google Drive (this may take a minute)...")
shutil.copy2('/content/gemma2-mi-therapist.gguf', DRIVE_OUTPUT)
print(f"Saved to: {DRIVE_OUTPUT}")
print("\nNow download this file from Google Drive to your laptop.")
print("Then follow the Ollama setup instructions in the next cell.")

## Step 6: Ollama Setup (Run on Your Laptop)

After downloading `gemma2-mi-therapist.gguf` to your laptop:

### 1. Install Ollama
Download from https://ollama.com and install it.

### 2. Create a Modelfile
Create a file called `Modelfile` in the same folder as the GGUF:

```
FROM ./gemma2-mi-therapist.gguf

PARAMETER temperature 0.7
PARAMETER top_p 0.9
PARAMETER repeat_penalty 1.2
PARAMETER num_predict 200

TEMPLATE """<start_of_turn>user
{{ .Prompt }}<end_of_turn>
<start_of_turn>model
{{ .Response }}<end_of_turn>
"""
```

### 3. Create the Ollama model
```bash
ollama create mi-therapist -f Modelfile
```

### 4. Test it
```bash
ollama run mi-therapist "I don't think I have a problem with drinking."
```

### 5. Update your chatbot config
In your `.env` file, set:
```
OLLAMA_MODEL=mi-therapist
```

The Streamlit app will now use your fine-tuned model!